In [1]:
%pip install neo4j
%pip install pykeen
%pip install torch torchvision

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 313 kB 977 kB/s eta 0:00:01
     |████████████████████████████████| 509 kB 2.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 718 kB 954 kB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 897 kB/s eta 0:00:01
     |████████████████████████████████| 174 kB 1.2 MB/s eta 0:00:01
     |████████████████████████████████| 43 kB 4.6 MB/s eta 0:00:01
     |████████████████████████████████| 69 kB 4.5 MB/s eta 0:00:01
     |████████████████████████████████| 10.8 MB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 73.6 MB 1.1 MB/s eta 0:00:011
     |█████████████████████████████

In [4]:
import torch
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
x = torch.rand(5, 3, device=device)  # Create tensor directly on GPU
print(x)

tensor([[0.6156, 0.0354, 0.5189],
        [0.2087, 0.8921, 0.3914],
        [0.9495, 0.1750, 0.7628],
        [0.7076, 0.6126, 0.8062],
        [0.9272, 0.6275, 0.7489]], device='mps:0')


In [5]:
from pykeen.utils import set_random_seed
set_random_seed(0)

/Users/khanjanin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/khanjanin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(None, <torch._C.Generator at 0x1112d59d0>, None)

In [6]:
import random
import numpy as np
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [7]:
# Define Neo4j connections
from neo4j import GraphDatabase
import pandas as pd
# import ampligraph

## Save the graph as csv and pickle

In [ ]:
# import the Neo4j driver
from neo4j import GraphDatabase
driver1 = GraphDatabase.driver(uri = "bolt://34.217.248.7:5080", auth=None)

In [12]:
def run_query(query, params={}):
    with driver1.session() as session:
        result = session.run(query, params)
        return pd.DataFrame([r.values() for r in result], columns=result.keys())

In [13]:
data = run_query(
"""
MATCH (s)-[r]->(t)
RETURN toString(id(s)) as source, toString(id(t)) AS target, type(r) as type
""")

data.head()
print(data[:10])

   source target                                  type
0  234104  17192          BRAINREGIONOVEREXPRESSESGENE
1  234229  17202          BRAINREGIONOVEREXPRESSESGENE
2  234038  17205         BRAINREGIONUNDEREXPRESSESGENE
3  233591  17276  TRANSCRIPTIONFACTORINTERACTSWITHGENE
4  234119  17276         BRAINREGIONUNDEREXPRESSESGENE
5  232420  17887             BODYPARTOVEREXPRESSESGENE
6  232932  17887             BODYPARTOVEREXPRESSESGENE
7  232371  18695             BODYPARTOVEREXPRESSESGENE
8  232411  18695            BODYPARTUNDEREXPRESSESGENE
9  232420  18695             BODYPARTOVEREXPRESSESGENE


In [ ]:
gene_gene_df =  run_query("""
MATCH (s)-[r:GENEINTERACTSWITHGENE|GENECOVARIESWITHGENE]->(t)
RETURN toString(id(t)) as source, toString(id(s)) AS target, type(r) as type
""")

gene_gene_df.head()

In [ ]:
combined_data = pd.concat([data, gene_gene_df], ignore_index=True)

In [ ]:
combined_data.shape

In [ ]:
combined_data.to_csv('alzkb_triples_memgraph.csv', sep='\t', index=False)

In [ ]:
import pickle
file_path_pickle = 'alzkb_triples_memgraph.pkl'
combined_data.to_pickle(file_path_pickle)